In [1]:
import os
import io
import zipfile
import requests
import pandas as pd
import time

from google.cloud import bigquery
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import (
    GeocoderRateLimited,
    GeocoderServiceError,
    GeocoderTimedOut
)

In [2]:
# --------------------------------------------------
# Configuration
# --------------------------------------------------

PROJECT_ID = "pacey32-agency"
client = bigquery.Client(
    project=PROJECT_ID
)

geolocator = Nominatim(
    user_agent="pacey32-hockey-cost-of-living",
    timeout=30
)

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=2.0,
    max_retries=2,
    error_wait_seconds=60,
    swallow_exceptions=False
)

# Local notebook testing only.
# Use an environment variable in GitHub Actions.
EIA_API_KEY = "8BxOAG5o9bKflmIbBUHbfvPsDqI4Gsx420oPj2os"

# Production version:
# EIA_API_KEY = os.environ.get("EIA_API_KEY")

In [3]:
# --------------------------------------------------
# Read NHL cities
# --------------------------------------------------

sql = """
SELECT DISTINCT
    venueLocation
FROM `pacey32-agency.Team.TeamList`
WHERE venueLocation IS NOT NULL
ORDER BY venueLocation
"""

cities = client.query(sql).to_dataframe()

print(f"Found {len(cities)} cities")

Found 32 cities


In [4]:
# --------------------------------------------------
# Identify Canadian cities
# --------------------------------------------------

canadian_cities = {
    "Calgary",
    "Edmonton",
    "Montreal",
    "Ottawa",
    "Toronto",
    "Vancouver",
    "Winnipeg"
}

In [5]:
# --------------------------------------------------
# Geocode cities
# --------------------------------------------------

location_results = []

for _, row in cities.iterrows():
    city = row["venueLocation"]
    country_hint = (
        "Canada"
        if city in canadian_cities
        else "United States"
    )
    query = f"{city}, {country_hint}"
    print(f"Geocoding {query}")

    try:
        location = geocode(
            query,
            addressdetails=True,
            language="en",
            exactly_one=True
        )

        if location is None:
            print(f"No result found for {city}")
            location_results.append({
                "venueLocation": city,
                "latitude": None,
                "longitude": None,
                "country": None,
                "country_code": None,
                "state_province": None,
                "state_code": None
            })

            continue

        address = location.raw.get(
            "address",
            {}
        )

        country_code = (
            address.get("country_code")
            or ""
        ).upper()

        state_province = (
            address.get("state")
            or address.get("province")
            or address.get("region")
        )

        state_code = (
            address.get("ISO3166-2-lvl4")
            or address.get("ISO3166-2-lvl3")
            or address.get("ISO3166-2-lvl6")
        )

        if state_code and "-" in state_code:
            state_code = state_code.split("-")[-1]
        location_results.append({
            "venueLocation": city,
            "latitude": float(location.latitude),
            "longitude": float(location.longitude),
            "country": address.get("country"),
            "country_code": country_code,
            "state_province": state_province,
            "state_code": state_code
        })

        print(
            f"Found {city}: "
            f"{state_province}, {state_code}, {country_code}"
        )

    except GeocoderRateLimited as error:
        print(f"Rate limited while geocoding {city}: {error}")
        location_results.append({
            "venueLocation": city,
            "latitude": None,
            "longitude": None,
            "country": None,
            "country_code": None,
            "state_province": None,
            "state_code": None
        })
        print("Waiting 120 seconds before continuing")
        time.sleep(120)

    except (
        GeocoderTimedOut,
        GeocoderServiceError
    ) as error:
        print(f"Geocoding failed for {city}: {error}")
        location_results.append({
            "venueLocation": city,
            "latitude": None,
            "longitude": None,
            "country": None,
            "country_code": None,
            "state_province": None,
            "state_code": None
        })

Geocoding Anaheim, United States
Found Anaheim: California, CA, US
Geocoding Boston, United States
Found Boston: Massachusetts, MA, US
Geocoding Buffalo, United States
Found Buffalo: New York, NY, US
Geocoding Calgary, Canada
Found Calgary: Alberta, AB, CA
Geocoding Chicago, United States
Found Chicago: Illinois, IL, US
Geocoding Columbus, United States
Found Columbus: Ohio, OH, US
Geocoding Dallas, United States
Found Dallas: Texas, TX, US
Geocoding Denver, United States
Found Denver: Colorado, CO, US
Geocoding Detroit, United States
Found Detroit: Michigan, MI, US
Geocoding Edmonton, Canada
Found Edmonton: Alberta, AB, CA
Geocoding Elmont, United States
Found Elmont: New York, NY, US
Geocoding Los Angeles, United States
Found Los Angeles: California, CA, US
Geocoding Montreal, Canada
Found Montreal: Quebec, QC, CA
Geocoding Nashville, United States
Found Nashville: Tennessee, TN, US
Geocoding New York, United States
Found New York: New York, NY, US
Geocoding Newark, United States
Fou

In [7]:
# --------------------------------------------------
# Create location dataframe
# --------------------------------------------------

city_locations = pd.DataFrame(
    location_results
)

In [ ]:
# --------------------------------------------------
# Create working dataframe
# --------------------------------------------------

cost_of_living = city_locations.copy()

In [ ]:
# --------------------------------------------------
# Download US Electricity Price Data from EIA API
# --------------------------------------------------

if not EIA_API_KEY:
    raise ValueError("EIA_API_KEY has not been set")

eia_electricity_url = (
    "https://api.eia.gov/v2/electricity/retail-sales/data/"
)

eia_electricity_params = {
    "api_key": EIA_API_KEY,
    "frequency": "monthly",
    "data[0]": "price",
    "facets[sectorid][]": "RES",
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
    "offset": 0,
    "length": 5000
}

response = requests.get(
    eia_electricity_url,
    params=eia_electricity_params,
    timeout=120
)

response.raise_for_status()

eia_json = response.json()

us_electricity_raw = pd.DataFrame(
    eia_json["response"]["data"]
)

us_electricity = us_electricity_raw.copy()

us_electricity["electricity_price"] = pd.to_numeric(
    us_electricity["price"],
    errors="coerce"
)

us_electricity["electricity_period"] = pd.to_datetime(
    us_electricity["period"],
    format="%Y-%m",
    errors="coerce"
)

us_electricity = (
    us_electricity
    .dropna(
        subset=[
            "stateid",
            "electricity_price",
            "electricity_period"
        ]
    )
    .sort_values(
        "electricity_period",
        ascending=False
    )
    .drop_duplicates(
        subset=["stateid"],
        keep="first"
    )
    .rename(
        columns={
            "stateid": "state_code"
        }
    )
    [
        [
            "state_code",
            "electricity_price",
            "electricity_period"
        ]
    ]
)

us_electricity["country_code"] = "US"
us_electricity["electricity_currency"] = "USD"
us_electricity["electricity_unit"] = "US cents per kWh"
us_electricity["electricity_geography_level"] = "State"
us_electricity["electricity_source"] = "US EIA"
us_electricity["electricity_source_table"] = (
    "Electricity retail sales"
)

,state_code,electricity_price,electricity_period,country_code,electricity_currency,electricity_unit,electricity_geography_level,electricity_source,electricity_source_table
0,AK,28.23,2026-05-01,US,USD,US cents per kWh,State,US EIA,Electricity retail sales
48,SC,16.18,2026-05-01,US,USD,US cents per kWh,State,US EIA,Electricity retail sales
34,NEW,28.14,2026-05-01,US,USD,US cents per kWh,State,US EIA,Electricity retail sales
35,NH,27.33,2026-05-01,US,USD,US cents per kWh,State,US EIA,Electricity retail sales
36,NJ,23.27,2026-05-01,US,USD,US cents per kWh,State,US EIA,Electricity retail sales


In [22]:
# --------------------------------------------------
# Download Canadian Electricity Price Data
# Source: Canada Energy Regulator
# --------------------------------------------------

canada_electricity_raw = pd.DataFrame([
    {
        "state_code": "AB",
        "electricity_price": 22.9,
        "electricity_period": "2025-11",
    },
    {
        "state_code": "BC",
        "electricity_price": 12.6,
        "electricity_period": "2025-11",
    },
    {
        "state_code": "MB",
        "electricity_price": 10.2,
        "electricity_period": "2025-11",
    },
    {
        "state_code": "ON",
        "electricity_price": 16.0,
        "electricity_period": "2025-11",
    },
    {
        "state_code": "QC",
        "electricity_price": 8.3,
        "electricity_period": "2025-11",
    }
])

canada_electricity = canada_electricity_raw.copy()

canada_electricity["electricity_price"] = pd.to_numeric(
    canada_electricity["electricity_price"],
    errors="coerce"
)

canada_electricity["electricity_period"] = pd.to_datetime(
    canada_electricity["electricity_period"],
    format="%Y-%m",
    errors="coerce"
)

canada_electricity = (
    canada_electricity
    .dropna(
        subset=[
            "state_code",
            "electricity_price",
            "electricity_period"
        ]
    )
)

canada_electricity["country_code"] = "CA"
canada_electricity["electricity_currency"] = "CAD"
canada_electricity["electricity_unit"] = "Canadian cents per kWh"
canada_electricity["electricity_geography_level"] = "Province"
canada_electricity["electricity_source"] = "Canada Energy Regulator"
canada_electricity["electricity_source_table"] = (
    "Average Residential Electricity Prices"
)

canada_electricity = canada_electricity[
    [
        "state_code",
        "electricity_price",
        "electricity_period",
        "country_code",
        "electricity_currency",
        "electricity_unit",
        "electricity_geography_level",
        "electricity_source",
        "electricity_source_table"
    ]
]

In [23]:
# --------------------------------------------------
# Convert states to team venues
# --------------------------------------------------

us_electricity = city_locations[
    city_locations["country_code"] == "US"
][
    [
        "venueLocation",
        "country_code",
        "state_code"
    ]
].merge(
    us_electricity,
    on=[
        "country_code",
        "state_code"
    ],
    how="left"
)

canada_electricity = city_locations[
    city_locations["country_code"] == "CA"
][
    [
        "venueLocation",
        "country_code",
        "state_code"
    ]
].merge(
    canada_electricity,
    on=[
        "country_code",
        "state_code"
    ],
    how="left"
)

In [25]:
# --------------------------------------------------
# Join US and Canadian electricity data
# --------------------------------------------------

electricity_df = pd.concat(
    [
        us_electricity,
        canada_electricity
    ],
    ignore_index=True
)

In [20]:
electricity = pd.concat(
    [us_electricity, canada_electricity],
    ignore_index=True
)

cost_of_living = city_locations.merge(
    electricity,
    on=["country_code", "state_code"],
    how="left"
)

In [26]:
cost_of_living = city_locations.copy()

for df in [
    electricity_df,
#    fuel_df,
#    income_tax_df,
#    sales_tax_df,
#    housing_df,
#    grocery_df,
#    transport_df
]:
    cost_of_living = cost_of_living.merge(
        df.drop(
            columns=[
                "country_code",
                "state_code"
            ]
        ),
        on="venueLocation",
        how="left"
    )

In [27]:
cost_of_living.head(20)

,venueLocation,latitude,longitude,country,country_code,state_province,state_code,electricity_price,electricity_period,electricity_currency,electricity_unit,electricity_geography_level,electricity_source,electricity_source_table
0,Anaheim,33.834752,-117.911732,United States,US,California,CA,33.25,2026-05-01,USD,US cents per kWh,State,US EIA,Electricity retail sales
1,Boston,42.358834,-71.057830,United States,US,Massachusetts,MA,28.82,2026-05-01,USD,US cents per kWh,State,US EIA,Electricity retail sales
2,Buffalo,42.886416,-78.878149,United States,US,New York,NY,29.93,2026-05-01,USD,US cents per kWh,State,US EIA,Electricity retail sales
3,Calgary,51.045606,-114.057541,Canada,CA,Alberta,AB,22.90,2025-11-01,CAD,Canadian cents per kWh,Province,Canada Energy Regulator,Average Residential Electricity Prices
4,Chicago,41.875562,-87.624421,United States,US,Illinois,IL,23.85,2026-05-01,USD,US cents per kWh,State,US EIA,Electricity retail sales
5,Columbus,39.962260,-83.000707,United States,US,Ohio,OH,19.52,2026-05-01,USD,US cents per kWh,State,US EIA,Electricity retail sales
6,Dallas,32.776272,-96.796856,United States,US,Texas,TX,16.44,2026-05-01,USD,US cents per kWh,State,US EIA,Electricity retail sales
7,Denver,39.739236,-104.984862,United States,US,Colorado,CO,16.16,2026-05-01,USD,US cents per kWh,State,US EIA,Electricity retail sales
8,Detroit,42.331551,-83.046640,United States,US,Michigan,MI,22.01,2026-05-01,USD,US cents per kWh,State,US EIA,Electricity retail sales
9,Edmonton,53.546205,-113.491241,Canada,CA,Alberta,AB,22.90,2025-11-01,CAD,Canadian cents per kWh,Province,Canada Energy Regulator,Average Residential Electricity Prices
